In [1]:
import pandas as pd
import pdfplumber

In [2]:
PDF_PATH = 'workers-compensation-premium-rates-2025-2026.pdf'

pdf = pdfplumber.open(PDF_PATH)
print(f'Total pages: {len(pdf.pages)}')

Total pages: 15


In [3]:
# Extract all data rows from tables across every page.
# Pages 1-14 have 4-column tables: [WIC Code, Description, WIC Rate, Dust Disease Rate]
# Page 15 has an 8-column table with None padding — same data, different layout.
# One row on page 15 has the WIC code shifted into column 1 instead of 0.
#
# 12 WIC codes use flat rates ($ per plate/mount/bout) instead of percentages:
# taxi drivers, jockeys, boxers, wrestlers, combat sports. These are stored
# in a separate column (flat_rate) with wic_rate_pct and dust_disease_rate_pct as NaN.

HEADER_VALUES = {'Column 1', 'Column 2', 'Column 3', 'Column 4',
                 'WIC Code', 'WIC Description', 'WIC Rate',
                 'Dust Diseases Rate', '(incl. GST)', None}

rows = []
for page in pdf.pages:
    for table in page.extract_tables():
        for row in table:
            # Skip header rows and the small dust-disease-only side table
            if row[0] in HEADER_VALUES and (len(row) == 1 or row[1] in HEADER_VALUES):
                continue

            # Normalise: extract meaningful values regardless of column count
            values = [v for v in row if v is not None and v.strip() != '']

            if len(values) == 4:
                wic_code, description, rate_str, dust_str = values
                wic_code = wic_code.strip()
                description = description.strip()

                if '%' in rate_str:
                    rows.append({
                        'wic_code': wic_code,
                        'description': description,
                        'wic_rate_pct': float(rate_str.replace('%', '')),
                        'dust_disease_rate_pct': float(dust_str.replace('%', '')),
                        'flat_rate': None,
                    })
                else:
                    # Flat-rate WICs (taxi plates, jockeys, boxers etc.)
                    rows.append({
                        'wic_code': wic_code,
                        'description': description,
                        'wic_rate_pct': None,
                        'dust_disease_rate_pct': None,
                        'flat_rate': rate_str.replace('\n', ' ').strip(),
                    })
            elif len(values) == 3:
                # Flat-rate rows only have 3 values (no dust disease rate)
                wic_code, description, rate_str = values
                rows.append({
                    'wic_code': wic_code.strip(),
                    'description': description.strip(),
                    'wic_rate_pct': None,
                    'dust_disease_rate_pct': None,
                    'flat_rate': rate_str.replace('\n', ' ').strip(),
                })
            else:
                print(f'Skipping unexpected row ({len(values)} values): {values}')
                continue

df = pd.DataFrame(rows)
print(f'{len(df)} rows parsed ({df["flat_rate"].notna().sum()} flat-rate, {df["wic_rate_pct"].notna().sum()} percentage-rate)')
df.head()

538 rows parsed (12 flat-rate, 526 percentage-rate)


,wic_code,description,wic_rate_pct,dust_disease_rate_pct,flat_rate
0,011100,Plant Nurseries,4.84,0.022,NaN
1,011200,Cut Flower and Flower Seed Growing,4.39,0.004,NaN
2,011300,Vegetable Growing,5.09,0.022,NaN
3,011400,Grape Growing,4.96,0.004,NaN
4,011500,Apple and Pear Growing,5.09,0.004,NaN


In [4]:
# Percentage-rate stats
df[df['wic_rate_pct'].notna()][['wic_rate_pct', 'dust_disease_rate_pct']].describe()

,wic_rate_pct,dust_disease_rate_pct
count,526.000000,526.000000
mean,3.717694,0.068110
std,2.476027,0.175832
min,0.213000,0.004000
25%,1.760000,0.004000
50%,3.420000,0.022000
75%,4.960000,0.066000
max,15.490000,1.815000


In [5]:
# Flat-rate WICs (taxi drivers, jockeys, boxers etc.)
df[df['flat_rate'].notna()][['wic_code', 'description', 'flat_rate']]

,wic_code,description,flat_rate
357,612310,Taxi Drivers - Metropolitan - T-Plate,"$1,977 per plate"
358,612315,Taxi Drivers - Metropolitan - T-Plate (up to 2...,"$1,158 per plate"
359,612320,Taxi Drivers - Non-Metropolitan - TC-Plate,"$1,575 per plate"
360,612322,Taxi Drivers - Non-Metropolitan - TC-plate (no...,$155 per plate
361,612324,Taxi Drivers - Non-Metropolitan - TC-plate (up...,$495 per plate
362,612326,Taxi Drivers - Non-Metropolitan - TC-plate (up...,$903 per plate
363,612330,Hire Car Drivers,"$1,285 per plate"
503,931120,Horse Racing Jockeys,$16 per mount or drive
504,931130,Horse Racing Harness Drivers,$16 per mount or drive
509,931930,Professional Boxers,$94 per capita per bout


In [6]:
# Spot-check first and last WIC codes, and the tricky page-15 rows
spot_checks = ['011100', '970000', '963100', '952200', '786100']
df[df['wic_code'].isin(spot_checks)][['wic_code', 'description', 'wic_rate_pct']]

,wic_code,description,wic_rate_pct
0,011100,Plant Nurseries,4.84
442,786100,Employment Placement Services,1.38
521,952200,Photographic Film Processing,1.29
533,963100,Police Services,5.22
537,970000,Private Households Employing Staff,1.80


In [8]:
df.shape

(538, 5)

In [ ]:
# Export to Parquet
df.to_parquet('NSW_WIC.parquet', index=False)
print('Saved NSW_WIC.parquet')

Saved NSW_WIC.csv
Saved NSW_WIC.parquet
